# Dark Manifold V29.4: Proper Michaelis-Menten Kinetics

V29.3 failed because reactions weren't properly coupled to substrate availability.
ATP accumulated infinitely because production wasn't limited by substrate depletion.

**Key fix**: Each reaction's flux depends on its ACTUAL substrates via proper MM kinetics.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Metabolic Network with Explicit Substrate Coupling

In [ ]:
class GlycolysisNetwork:
    """Glycolysis with proper substrate-product relationships."""
    
    def __init__(self):
        self.metabolites = [
            'glc',    # 0: Glucose (external)
            'g6p',    # 1: Glucose-6-phosphate
            'f6p',    # 2: Fructose-6-phosphate
            'fbp',    # 3: Fructose-1,6-bisphosphate
            'gap',    # 4: Glyceraldehyde-3-P (represents trioses)
            'bpg',    # 5: 1,3-bisphosphoglycerate
            'pep',    # 6: Phosphoenolpyruvate (represents 3PG+2PG+PEP)
            'pyr',    # 7: Pyruvate
            'lac',    # 8: Lactate
            'atp',    # 9: ATP
            'adp',    # 10: ADP
            'nad',    # 11: NAD+
            'nadh',   # 12: NADH
            'pi',     # 13: Phosphate
        ]
        self.n_met = len(self.metabolites)
        self.met_idx = {m: i for i, m in enumerate(self.metabolites)}
        
        # Each reaction: (name, substrates, products, kcat, Km)
        # Substrates/products are lists of (metabolite_idx, stoich_coeff)
        self.reactions = [
            # HK: glc + atp -> g6p + adp
            ('HK', [(0, 1), (9, 1)], [(1, 1), (10, 1)], 100.0, 0.1),
            # PGI: g6p <-> f6p
            ('PGI', [(1, 1)], [(2, 1)], 500.0, 0.5),
            # PFK: f6p + atp -> fbp + adp
            ('PFK', [(2, 1), (9, 1)], [(3, 1), (10, 1)], 100.0, 0.1),
            # ALDO: fbp -> 2 gap
            ('ALDO', [(3, 1)], [(4, 2)], 50.0, 0.05),
            # GAPDH: gap + nad + pi -> bpg + nadh
            ('GAPDH', [(4, 1), (11, 1), (13, 1)], [(5, 1), (12, 1)], 100.0, 0.1),
            # PGK: bpg + adp -> pep + atp  (lumped PGK+PGM+ENO)
            ('PGK', [(5, 1), (10, 1)], [(6, 1), (9, 1)], 500.0, 0.1),
            # PK: pep + adp -> pyr + atp
            ('PK', [(6, 1), (10, 1)], [(7, 1), (9, 1)], 200.0, 0.1),
            # LDH: pyr + nadh -> lac + nad
            ('LDH', [(7, 1), (12, 1)], [(8, 1), (11, 1)], 300.0, 0.2),
            # ATPase: atp -> adp + pi (cellular ATP consumption)
            ('ATPase', [(9, 1)], [(10, 1), (13, 1)], 100.0, 1.0),
        ]
        self.n_rxn = len(self.reactions)
        
        # Build stoichiometry matrix
        self.S = np.zeros((self.n_met, self.n_rxn))
        self.kcat = np.zeros(self.n_rxn)
        self.Km = np.zeros(self.n_rxn)
        
        for j, (name, subs, prods, kcat, km) in enumerate(self.reactions):
            self.kcat[j] = kcat
            self.Km[j] = km
            for met_idx, coef in subs:
                self.S[met_idx, j] = -coef
            for met_idx, coef in prods:
                self.S[met_idx, j] = coef
        
        # Initial concentrations (mM) - physiological values
        self.M0 = np.array([
            5.0,     # glc: 5 mM (blood glucose level)
            0.1,     # g6p
            0.05,    # f6p
            0.02,    # fbp
            0.05,    # gap
            0.001,   # bpg (very low, transient)
            0.1,     # pep
            0.1,     # pyr
            1.0,     # lac
            3.0,     # atp: ~3 mM
            0.5,     # adp: ~0.5 mM
            1.0,     # nad
            0.1,     # nadh
            5.0,     # pi
        ])
        
        print(f"Network: {self.n_met} metabolites, {self.n_rxn} reactions")
        print(f"ATP/ADP ratio: {self.M0[9]/self.M0[10]:.1f}")

network = GlycolysisNetwork()

In [ ]:
# Verify mass balance for ATP
print("ATP stoichiometry check:")
atp_idx = network.met_idx['atp']
for j, (name, _, _, _, _) in enumerate(network.reactions):
    coef = network.S[atp_idx, j]
    if coef != 0:
        sign = "+" if coef > 0 else ""
        print(f"  {name}: {sign}{coef:.0f} ATP")

print("\nExpected balance per glucose:")
print("  HK: -1 ATP")
print("  PFK: -1 ATP") 
print("  PGK: +2 ATP (×2 trioses)")
print("  PK: +2 ATP (×2 trioses)")
print("  Net: +2 ATP per glucose (correct for glycolysis)")

## 2. Model with Proper Substrate-Coupled Kinetics

In [ ]:
class ProperMMModel(nn.Module):
    """
    Model with PROPER Michaelis-Menten kinetics.
    
    Key: Each reaction's rate depends on ALL its substrates.
    v = kcat * product_i(S_i / (Km_i + S_i))
    
    This ensures reactions STOP when substrates are depleted.
    """
    
    def __init__(self, network, hidden_dim=64):
        super().__init__()
        
        self.n_met = network.n_met
        self.n_rxn = network.n_rxn
        
        # Register network data
        self.register_buffer('S', torch.tensor(network.S, dtype=torch.float32))
        self.register_buffer('kcat', torch.tensor(network.kcat, dtype=torch.float32))
        self.register_buffer('Km', torch.tensor(network.Km, dtype=torch.float32))
        self.register_buffer('M0', torch.tensor(network.M0, dtype=torch.float32))
        
        # Store substrate indices for each reaction
        # This is the KEY to proper MM kinetics
        self.substrate_info = []
        for j, (name, subs, prods, kcat, km) in enumerate(network.reactions):
            self.substrate_info.append({
                'name': name,
                'substrate_indices': [s[0] for s in subs],
                'substrate_stoich': [s[1] for s in subs],
            })
        
        # Neural network for regulatory effects (small corrections only)
        self.reg_net = nn.Sequential(
            nn.Linear(self.n_met, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, self.n_rxn),
            nn.Tanh(),  # Output in [-1, 1]
        )
        
        # Initialize to zero (no correction initially)
        nn.init.zeros_(self.reg_net[-2].weight)
        nn.init.zeros_(self.reg_net[-2].bias)
        
        # Learnable regulation strength (starts small)
        self.reg_strength = nn.Parameter(torch.tensor(0.1))
    
    def compute_mm_flux(self, M):
        """
        Proper Michaelis-Menten kinetics.
        
        For each reaction j:
        v_j = kcat_j * product over substrates i of (M_i / (Km + M_i))
        
        This PROPERLY couples flux to substrate availability.
        """
        batch_size = M.shape[0]
        v = torch.zeros(batch_size, self.n_rxn, device=M.device)
        
        for j, info in enumerate(self.substrate_info):
            # Start with kcat
            flux = self.kcat[j]
            
            # Multiply by saturation term for EACH substrate
            for sub_idx in info['substrate_indices']:
                S = M[:, sub_idx]  # Substrate concentration
                Km = self.Km[j]
                sat = S / (Km + S + 1e-8)  # Saturation: 0 to 1
                flux = flux * sat
            
            v[:, j] = flux
        
        return v
    
    def forward(self, M, dt=0.001):
        """
        One integration step.
        """
        # Ensure positive concentrations
        M = torch.clamp(M, min=1e-8)
        
        # Base flux from proper MM kinetics
        v_mm = self.compute_mm_flux(M)
        
        # Small neural correction for regulatory effects
        M_norm = torch.log(M + 1e-6) / 3.0  # Normalize in log space
        reg = self.reg_net(M_norm)  # In [-1, 1]
        
        # Apply small correction: multiply by (1 + strength * reg)
        # This keeps corrections bounded: flux * [0.9, 1.1] roughly
        correction = 1.0 + self.reg_strength.abs() * reg
        correction = torch.clamp(correction, min=0.5, max=2.0)
        
        v = v_mm * correction
        
        # Rate of change: dM/dt = S @ v
        dM_dt = torch.matmul(v, self.S.T)
        
        # Euler integration
        M_next = M + dM_dt * dt
        
        # Clamp to physical bounds
        M_next = torch.clamp(M_next, min=1e-8, max=100.0)
        
        return M_next, v
    
    def simulate(self, n_steps, dt=0.001, save_every=1):
        """Run simulation."""
        M = self.M0.unsqueeze(0)
        
        n_save = n_steps // save_every + 1
        M_hist = torch.zeros(n_save, self.n_met, device=M.device)
        v_hist = torch.zeros(n_save, self.n_rxn, device=M.device)
        
        M_hist[0] = M[0]
        idx = 0
        
        with torch.no_grad():
            for step in range(n_steps):
                M, v = self.forward(M, dt)
                if (step + 1) % save_every == 0:
                    idx += 1
                    if idx < n_save:
                        M_hist[idx] = M[0]
                        v_hist[idx] = v[0]
        
        return {
            'M': M_hist.cpu().numpy(),
            'v': v_hist.cpu().numpy(),
            'time_min': np.arange(n_save) * save_every * dt,
        }

model = ProperMMModel(network).to(device)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Test: verify MM kinetics work correctly
print("Testing MM kinetics...")
model.eval()
with torch.no_grad():
    M_test = model.M0.unsqueeze(0)
    v = model.compute_mm_flux(M_test)
    print("\nFlux at initial conditions:")
    for j, info in enumerate(model.substrate_info):
        print(f"  {info['name']:8s}: {v[0, j].item():8.2f} mM/min")
    
    # Test with depleted ATP
    M_low_atp = M_test.clone()
    M_low_atp[0, network.met_idx['atp']] = 0.01  # Very low ATP
    v_low = model.compute_mm_flux(M_low_atp)
    print("\nFlux with low ATP (0.01 mM):")
    for j, info in enumerate(model.substrate_info):
        if 'atp' in [network.metabolites[i] for i in info['substrate_indices']]:
            print(f"  {info['name']:8s}: {v_low[0, j].item():8.2f} mM/min (should be ~0)")

## 3. Generate Training Data

In [ ]:
def generate_data(network, n_traj=100, duration=5.0, dt=0.01):
    """Generate trajectories using ODE solver with proper MM kinetics."""
    from scipy.integrate import odeint
    
    def mm_flux(M, network):
        """Compute flux with proper MM kinetics."""
        M = np.maximum(M, 1e-8)
        v = np.zeros(network.n_rxn)
        for j, (name, subs, prods, kcat, km) in enumerate(network.reactions):
            flux = kcat
            for sub_idx, _ in subs:
                S = M[sub_idx]
                flux *= S / (km + S)
            v[j] = flux
        return v
    
    def ode_rhs(M, t):
        v = mm_flux(M, network)
        dM_dt = network.S @ v
        return dM_dt
    
    trajectories = []
    t = np.arange(0, duration, dt)
    
    for _ in tqdm(range(n_traj), desc="Generating"):
        # Perturb initial conditions
        M0 = network.M0 * np.exp(np.random.randn(network.n_met) * 0.3)
        M0 = np.clip(M0, 1e-4, 50)
        
        try:
            sol = odeint(ode_rhs, M0, t)
            if np.all(np.isfinite(sol)) and np.all(sol > 0):
                trajectories.append({'M': sol, 't': t})
        except:
            pass
    
    print(f"Generated {len(trajectories)} valid trajectories")
    return trajectories

train_data = generate_data(network, n_traj=200)

In [ ]:
# Visualize one trajectory to verify
if train_data:
    traj = train_data[0]
    t = traj['t']
    M = traj['M']
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    ax = axes[0]
    ax.plot(t, M[:, network.met_idx['atp']], 'b-', label='ATP')
    ax.plot(t, M[:, network.met_idx['adp']], 'r-', label='ADP')
    ax.set_xlabel('Time (min)')
    ax.set_ylabel('Concentration (mM)')
    ax.set_title('Energy carriers (ODE reference)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    ax = axes[1]
    ax.plot(t, M[:, network.met_idx['glc']], label='Glucose')
    ax.plot(t, M[:, network.met_idx['pyr']], label='Pyruvate')
    ax.plot(t, M[:, network.met_idx['lac']], label='Lactate')
    ax.set_xlabel('Time (min)')
    ax.set_ylabel('Concentration (mM)')
    ax.set_title('Metabolites (ODE reference)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"ATP at end: {M[-1, network.met_idx['atp']]:.2f} mM")
    print(f"Glucose consumed: {M[0, network.met_idx['glc']] - M[-1, network.met_idx['glc']]:.2f} mM")

## 4. Training

In [ ]:
def train(model, data, epochs=500, lr=1e-3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    
    history = {'loss': []}
    best_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        n_batch = 0
        
        for traj in data:
            M_true = torch.tensor(traj['M'], dtype=torch.float32, device=device)
            seq_len = min(len(M_true) - 1, 50)
            
            # One-step prediction
            M_in = M_true[:seq_len]
            M_target = M_true[1:seq_len+1]
            
            M_pred, _ = model(M_in, dt=0.01)
            
            # MSE loss in log space
            loss = torch.mean((torch.log(M_pred + 1e-6) - torch.log(M_target + 1e-6))**2)
            
            if torch.isfinite(loss):
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                total_loss += loss.item()
                n_batch += 1
        
        scheduler.step()
        avg_loss = total_loss / max(n_batch, 1)
        history['loss'].append(avg_loss)
        
        if avg_loss < best_loss and np.isfinite(avg_loss):
            best_loss = avg_loss
            torch.save(model.state_dict(), 'best_v29_4.pt')
        
        if (epoch + 1) % 50 == 0:
            model.eval()
            with torch.no_grad():
                result = model.simulate(1000, dt=0.01, save_every=10)
                atp = result['M'][:, network.met_idx['atp']]
                adp = result['M'][:, network.met_idx['adp']]
            print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | ATP: {atp[-1]:.2f} | ADP: {adp[-1]:.2f}")
    
    return history

print("Training...")
history = train(model, train_data, epochs=500)

## 5. Full 20-Minute Simulation

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_v29_4.pt', map_location=device, weights_only=True))
model.eval()

# Compile for speed (PyTorch 2.0+)
try:
    model = torch.compile(model, mode='reduce-overhead')
    print('Model compiled for faster execution!')
except:
    print('torch.compile not available, using eager mode')

print('\n' + '='*60)
print('REAL-TIME WHOLE-CELL SIMULATION')
print('='*60)
print(f'Target: 20 virtual minutes')
print(f'Resolution: 1 millisecond (1,200,000 steps)')
print(f'Must complete in <20 real minutes for real-time')
print('='*60 + '\n')

# Simulation parameters
DURATION_MIN = 20
DT_MS = 1  # 1 millisecond
SAVE_EVERY_MS = 100  # Save every 100ms for smooth playback
N_STEPS = DURATION_MIN * 60 * 1000
N_SAVE = N_STEPS // SAVE_EVERY_MS

print(f'Total steps: {N_STEPS:,}')
print(f'Saved frames: {N_SAVE:,} (one per 100ms)')
print()

# Real-time simulation with progress tracking
start_time = time.time()

M = model.M0.unsqueeze(0).to(device)
M_history = torch.zeros(N_SAVE + 1, model.n_met, device=device)
M_history[0] = M[0]

save_idx = 0
last_print = 0

print('Simulating... (virtual time vs real time)')
print('-' * 60)

with torch.no_grad():
    for step in range(N_STEPS):
        M, _ = model.forward(M, dt=DT_MS/60000)
        
        if (step + 1) % SAVE_EVERY_MS == 0:
            save_idx += 1
            M_history[save_idx] = M[0]
            
            # Progress update every simulated minute
            virtual_min = (step + 1) * DT_MS / 60000
            if virtual_min >= last_print + 1:
                real_sec = time.time() - start_time
                speed = virtual_min / (real_sec / 60) if real_sec > 0 else 0
                atp_now = M[0, network.met_idx['atp']].item()
                print(f'  Virtual: {virtual_min:5.1f} min | Real: {real_sec:5.1f} sec | '
                      f'Speed: {speed:5.0f}x | ATP: {atp_now:.2f} mM')
                last_print = virtual_min

total_time = time.time() - start_time
print('-' * 60)
print(f'\n✅ SIMULATION COMPLETE!')
print(f'   20 virtual minutes completed in {total_time:.1f} real seconds')
print(f'   Speed: {DURATION_MIN * 60 / total_time:.0f}x faster than real-time')

if total_time < DURATION_MIN * 60:
    print(f'   ✓ REAL-TIME CAPABLE with {DURATION_MIN * 60 / total_time:.0f}x headroom')
else:
    print(f'   ✗ Slower than real-time')

# Convert to numpy for analysis
result = {
    'M': M_history.cpu().numpy(),
    'time_min': np.arange(N_SAVE + 1) * SAVE_EVERY_MS / 60000,
}

In [ ]:
# Visualize
M = result['M']
t = result['time_min']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# ATP/ADP
ax = axes[0, 0]
ax.plot(t, M[:, network.met_idx['atp']], 'b-', label='ATP', lw=0.5)
ax.plot(t, M[:, network.met_idx['adp']], 'r-', label='ADP', lw=0.5)
ax.axhline(3.0, color='b', ls='--', alpha=0.5, label='ATP target')
ax.set_xlabel('Time (min)')
ax.set_ylabel('Concentration (mM)')
ax.set_title('Energy Carriers')
ax.legend()
ax.grid(True, alpha=0.3)

# Glucose/Lactate
ax = axes[0, 1]
ax.plot(t, M[:, network.met_idx['glc']], label='Glucose', lw=0.5)
ax.plot(t, M[:, network.met_idx['pyr']], label='Pyruvate', lw=0.5)
ax.plot(t, M[:, network.met_idx['lac']], label='Lactate', lw=0.5)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Concentration (mM)')
ax.set_title('Glycolysis Products')
ax.legend()
ax.grid(True, alpha=0.3)

# Energy charge
atp = M[:, network.met_idx['atp']]
adp = M[:, network.met_idx['adp']]
ec = atp / (atp + adp + 0.01)

ax = axes[1, 0]
ax.plot(t, ec, 'purple', lw=0.5)
ax.axhline(0.85, color='g', ls='--', label='Optimal (0.85)')
ax.set_xlabel('Time (min)')
ax.set_ylabel('ATP/(ATP+ADP)')
ax.set_title('Energy Charge')
ax.set_ylim([0.5, 1.0])
ax.legend()
ax.grid(True, alpha=0.3)

# Training
ax = axes[1, 1]
ax.semilogy(history['loss'])
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('v29_4_results.png', dpi=150)
plt.show()

print(f"\nFinal state:")
print(f"  ATP: {atp[-1]:.2f} mM (target ~3)")
print(f"  ADP: {adp[-1]:.2f} mM")
print(f"  Energy charge: {ec[-1]:.3f}")
print(f"  Glucose: {M[0, network.met_idx['glc']]:.2f} -> {M[-1, network.met_idx['glc']]:.2f} mM")

In [ ]:
# Save
import json

output = {
    'metadata': {
        'model': 'Dark Manifold V29.4',
        'description': 'Proper Michaelis-Menten kinetics',
        'duration_min': 20,
        'resolution_ms': 100,
    },
    'time_min': t.tolist(),
    'metabolites': {name: M[:, i].tolist() for name, i in network.met_idx.items()},
    'energy_charge': ec.tolist(),
}

with open('whole_cell_v29_4.json', 'w') as f:
    json.dump(output, f)
    
torch.save(model.state_dict(), 'whole_cell_v29_4.pt')
print("Saved results!")